# Canonical MathLive rebuild showcase notebook — Phase 2

This notebook is the **primary evidence surface** for this phase.

It is intentionally saved **without outputs**. The point is for **you** to run it,
open the widget menus yourself, type into the fields yourself, and decide whether
what you see matches the written expectations.


## Phase goal

This phase is trying to prove five visible things:

1. the generic `MathInput` field still behaves like a generic expression field,
2. `IdentifierInput` is still a **separate** widget instead of a role flag on the
   generic field,
3. the identifier field uses an **explicit** context policy,
4. the identifier field menu is visibly more constrained than the generic field menu,
5. the identifier field now keeps the visible surface to the MathLive field itself.

This correction also targets two concrete failures from the previous attempt:

- the identifier widget should **not** trigger MathLive constructor-option warnings,
- the identifier widget should **not** render a suggestion strip below the field.


## How to use this notebook

Run the cells in order.

For each widget demo:

- read the goal first,
- interact with the field manually,
- open the field menu manually,
- compare what you see with the written expectations,
- rerun the nearby `...value` cell after each edit when you want to inspect the
  synchronized Python value,
- only then decide whether the checklist item should count as passing.

Do **not** treat a passing Python value check as proof that the UI is correct.
The menu contents and visible behavior are the main evidence in this phase.


In [1]:
import import_setup
from gu_toolkit import IdentifierInput, MathInput


## Context used in this phase

This phase uses one small explicit context:

- `mass`
- `time`
- `speed`

Why this matters:

- in `context_only`, only these names should be accepted,
- in `context_or_new`, these names should be **suggested through the identifier menu**,
  while a new plain identifier should still be allowed,
- the context is printed in the notebook so you can compare the printed list with
  the identifier field menu,
- the identifier widget itself should stay visually minimal: **only the MathLive
  field should be visible**.


In [2]:
identifier_context = ["mass", "time", "speed"]
identifier_context


['mass', 'time', 'speed']

## Demo 1 — generic expression field baseline

### What is this demo trying to prove?

The rebuild still includes a generic expression field that is **not** an
identifier-specific widget.

### What should happen?

The field should render as a normal MathLive expression editor.

When you open its menu, you may see broader expression-oriented items that make
sense for a general math editor. Depending on your runtime, these may include
things such as evaluate/simplify/solve commands, styling or background commands,
or other expression helpers.

### Why should it happen?

This field is the comparison point. The identifier field is supposed to be more
constrained than this one.

### What would indicate the implementation is still wrong?

Failure would include either of these:

- the generic field menu is already reduced to the same narrow identifier menu,
- the generic field unexpectedly shows identifier-specific context-name insertion.


In [4]:
generic_field = MathInput(value=r"\sin(x)+1")
generic_field


## Demo 2 — identifier field with `context_only`

### What is this demo trying to prove?

A separate identifier widget exists and is visibly more constrained than the
generic expression field.

### What should happen?

The field should render as a MathLive field, but its visible surface should stay
minimal:

- you should see **only the field itself**,
- you should **not** see a suggestion strip, chip list, helper row, or any other
  extra visible control under the field.

When you open the identifier field menu, you should see a constrained menu with:

- a `Context names` submenu containing `mass`, `time`, and `speed`,
- basic edit commands such as `Undo`, `Redo`, `Cut`, `Copy`, `Paste`, and
  `Select all`.

You should **not** see generic expression-only items here, such as solve/evaluate
commands, background or styling actions, or broad expression insertion helpers.

### Why should it happen?

This field is meant to act like a plain-identifier editor with an explicit
context rule, not like a full generic expression editor.

### What would indicate the implementation is still wrong?

Failure would include any of these:

- a suggestion strip appears below the field,
- the identifier menu still exposes solve/evaluate or styling/background items,
- the `Context names` submenu is missing or does not match the printed context.


In [5]:
context_only_field = IdentifierInput(
    value="mass",
    context_names=identifier_context,
    context_policy="context_only",
)
context_only_field


## Demo 3 — manual checks for `context_only`

### What is this demo trying to prove?

`context_only` should accept only names from the provided context, while keeping
invalid input visible enough for you to inspect it.

### What should happen?

Try these by hand in the widget above, rerunning the value cell after each step
when you want to inspect the backend value:

- type `time` — it should be accepted,
- type `angle` — it should remain visibly invalid for this policy,
- type `x+1` — it should remain visibly invalid,
- use the menu and pick one of the names from `Context names` — it should insert
  that accepted identifier.

The Python value cell below should continue to show the **last accepted** value
when the current visible draft is invalid.

### Why should it happen?

This is the simplest visible proof that the identifier field is enforcing an
explicit policy instead of behaving like a generic expression editor.

### What would indicate the implementation is still wrong?

Failure would include either of these:

- an unknown name such as `angle` is accepted under `context_only`,
- invalid content silently disappears or overwrites the synchronized Python value.


In [6]:
context_only_field.value


'mass'

## Demo 4 — identifier field with `context_or_new`

### What is this demo trying to prove?

The second explicit policy should still offer the provided context names in the
menu while also allowing a new plain identifier.

### What should happen?

The field should again show **only the MathLive field**.

Its menu should again contain the `Context names` submenu with `mass`, `time`,
and `speed`, plus the same small edit-command set.

Now try these by hand:

- type `speed` — accepted from context,
- type `angle` — accepted as a new plain identifier,
- type `x+1` — invalid,
- type `x_1` — invalid in this phase,
- type `sin(x)` — invalid,
- type `\alpha` — invalid in this phase.

### Why should it happen?

This phase allows exactly one small policy difference: provided context names
are still suggested, but a new plain identifier is allowed when it matches the
published simple pattern.

### What would indicate the implementation is still wrong?

Failure would include either of these:

- a new plain identifier such as `angle` is rejected under `context_or_new`,
- expression-shaped input such as `x+1` is treated like a valid identifier.


In [7]:
context_or_new_field = IdentifierInput(
    value="mass",
    context_names=identifier_context,
    context_policy="context_or_new",
)
context_or_new_field


## Demo 5 — plain-text `sin` and `log` check

### What is this demo trying to prove?

The identifier field should not behave like a generic function-entry surface.

### What should happen?

In the `context_or_new` widget above, type plain-text `sin` and plain-text `log`.

They should **not** auto-convert into built-in MathLive function commands.
In this phase, they should stay plain identifier text. Because `context_or_new`
allows new identifiers, they may be accepted as ordinary identifiers.

### Why should it happen?

This phase is supposed to remove identifier-irrelevant generic MathLive actions,
including the shortcut path that would turn ordinary letter sequences into
function input.

### What would indicate the implementation is still wrong?

Failure would include either of these:

- typing `sin` or `log` turns into function-style MathLive input,
- the field menu still offers direct function insertion commands for identifiers.


In [13]:
context_or_new_field.value


''

## Demo 6 — browser console check

### What is this demo trying to prove?

This correction should no longer rely on the invalid MathLive constructor-option
pattern used in the previous attempt.

### What should happen?

Open the browser developer console while the widgets above render.

You should **not** see MathLive warnings saying that options such as:

- `inlineShortcuts`
- `scriptDepth`
- `smartFence`
- `smartMode`
- `smartSuperscript`

cannot be used as constructor options.

### Why should it happen?

Those warnings indicate the identifier widget was still being built with an
unapproved constructor-options path instead of a simpler, auditable post-mount
configuration path.

### What would indicate the implementation is still wrong?

Any of those constructor-option warnings count as failure for this correction.


## Manual verification checklist

Only check an item after you have **personally observed** it in this notebook.

- [x] The generic `MathInput` field renders as a generic expression editor.
- [x] The generic field menu is accessible.
- [x] The identifier widget renders as a separate, visibly constrained field.
- [x] The identifier widget shows **only** the MathLive field itself.
- [x] There is **no** suggestion strip or chip list below the identifier field.
- [x] The identifier field menu contains a `Context names` submenu.
- [x] The `Context names` submenu matches the printed context exactly.
- [x] The identifier field menu keeps only the small edit-command set plus the
      context-name submenu.
- [x] The identifier field menu does **not** show solve/evaluate items.
- [x] The identifier field menu does **not** show styling/background items.
- [x] The `context_only` field accepts `mass`, `time`, and `speed`.
- [x] The `context_only` field rejects an unknown name such as `angle`.
- [x] Under invalid input, the visible field stays inspectable while the Python
      value cell keeps the last accepted identifier.
- [x] The `context_or_new` field accepts a new plain identifier such as `angle`.
- [x] The `context_or_new` field rejects expression-shaped input such as `x+1`.
- [x] Typing plain-text `sin` or `log` does not auto-convert into function input.
- [x] The browser console does not show MathLive constructor-option warnings for
      `inlineShortcuts`, `scriptDepth`, `smartFence`, `smartMode`, or
      `smartSuperscript`.


## Short report for this phase

### Implemented

- generic expression field retained as the comparison baseline,
- separate identifier field retained instead of role mutation,
- explicit `context_only` and `context_or_new` policies,
- menu-based context suggestions,
- identifier menu constrained to context insertion plus basic edit commands,
- identifier widget surface reduced to the MathLive field itself,
- post-mount identifier configuration instead of constructor-option injection.

### Not implemented yet

- broader `gu_toolkit.identifiers` semantics,
- underscores, subscripts, or superscripts as canonical identifier structure,
- Greek-name identifier policies,
- raw LaTeX identifier values,
- broad semantic intelligence.

### What you must verify manually

- compare the generic and identifier menus,
- confirm the identifier field has no visible suggestion strip,
- confirm the identifier menu contains the printed context names,
- confirm `context_only` and `context_or_new` differ exactly as described,
- confirm plain-text `sin` and `log` do not become function input,
- confirm the browser console is clean of the constructor-option warnings listed above.

### What counts as failure

- the identifier field still shows a suggestion strip below the widget,
- the identifier menu still exposes generic expression-only actions,
- the context submenu is missing or wrong,
- `context_only` accepts unknown names,
- `context_or_new` rejects a plain new identifier,
- plain-text `sin` or `log` auto-converts into function input,
- MathLive still reports invalid constructor options in the browser console.

**Ready for user verification.**
